# Perception Test の学習映像からフレームを抜いて Drive に置く

手元の回線では 26.5GB の取得に 8 時間かかるので、Colab で取得してフレーム（JPEG）だけ Drive に残す。GPU は不要。

手順:
1. ランタイム → ランタイムのタイプを変更 → **CPU**（ハイメモリでなくてよい）。GPU は使わないのでユニットを消費しない
2. 上から順に実行する。取得 + 展開 + フレーム抜き出しで 1〜2 時間
3. 終わると `MyDrive/movie_llm/pt_frames_f32_s640.zip`（数 GB）ができる


In [ ]:
!df -h /content | tail -1
!nproc

## 1. リポジトリと依存パッケージ

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
!pip install -q "av>=14" "pillow>=10" "numpy" "transformers>=5.0" "torch" --extra-index-url https://download.pytorch.org/whl/cpu
!pip install -q -e . --no-deps
import av; print("av", av.__version__)

## 2. Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
import os; os.makedirs(DRIVE, exist_ok=True)

## 3. 学習映像を取得して展開（26.5GB。Google のネットワーク内なので数分〜十数分）

In [ ]:
%cd /content
!curl -L -o train_videos.zip https://storage.googleapis.com/dm-perception-test/zip_data/train_videos.zip
!ls -la train_videos.zip

In [ ]:
# bzip2 圧縮の zip なので Python の zipfile で展開する
import zipfile, time
t0 = time.time()
with zipfile.ZipFile('/content/train_videos.zip') as z:
    z.extractall('/content/pt')
print(f'展開 {time.time()-t0:.0f}s')
!rm /content/train_videos.zip
!ls /content/pt | head; ls /content/pt/videos | wc -l; df -h /content | tail -1

## 4. 32 フレームを抜く（並列。1〜2 時間）

In [ ]:
%cd /content/movie_llm
import os
!python scripts/jvqa_cache_frames.py --frames 32 --video-dir /content/pt/videos --frames-dir /content/pt_frames --workers {os.cpu_count()}
!du -sh /content/pt_frames

## 5. zip にして Drive へ

In [ ]:
import shutil, time
t0 = time.time()
shutil.make_archive('/content/pt_frames_f32_s640', 'zip', '/content/pt_frames')
print(f'zip {time.time()-t0:.0f}s')
!ls -la /content/pt_frames_f32_s640.zip
!cp /content/pt_frames_f32_s640.zip {DRIVE}/
!ls -la {DRIVE}/

終わったら、手元の私に「フレームできた」と伝える。zip はダウンロードせず Drive に置いたままでよい（学習と問題の選別も Colab で行い、Drive から読む）。